# prahari — train the extraction model on Colab

This trains the **only** part of prahari that needs a GPU: the MuRIL + LoRA
token-classification model that extracts spans from report text.

It does **not** train a classifier. The model never emits a SIF verdict — it
finds `ENERGY_SOURCE`, `MAGNITUDE_CUE`, `CONTROL_MENTION`, `CONTROL_NEGATION`,
`ACTIVITY` and `LOCATION` spans, and the deterministic rule engine decides.
That split is the point of the system; nothing here changes it.

---

### Before you run anything

**Runtime → Change runtime type → T4 GPU.** On CPU this takes all day.

Total wall time on a free T4: roughly **60–110 minutes**, nearly all of it in
step 4. You can close the tab, but Colab disconnects idle sessions — check
back every ~20 minutes or the run dies and you start over.

### What you get at the end

A `prahari-models.zip` downloaded to your laptop. Unzip it into the repo's
`models/` directory and the backend switches from the keyword extractor to
the ONNX path automatically — the startup banner will say so.


## 1 · Confirm the GPU

If this errors or prints nothing, the runtime is still on CPU. Fix it before
going on — you will not enjoy discovering it in step 4.


In [ ]:
!nvidia-smi


## 2 · Clone the repo and install the training extras

`[train]` pulls torch, transformers, peft, accelerate and onnx. None of these
are runtime dependencies — a test in the repo fails the build if anything
under `prahari/` ever imports them at runtime.

Expect ~2 minutes and a pip resolver warning or two; those are harmless.


In [ ]:
!git clone --depth 1 https://github.com/adityacs50-lab/Antardrishti.git /content/prahari
%cd /content/prahari
!pip install -q -e ".[train]"
print('\nrepo + deps ready')


## 3 · Build the BIO dataset

Reads `gold_spans` from the synthetic corpus — what the generator *knows* it
wrote — and aligns them to MuRIL wordpieces.

The labels deliberately do **not** come from the keyword extractor. Training
on its matches would produce a model that can only ever imitate it, and the
whole reason to train is to generalise past it.

Takes under a minute. Check the printed split sizes look like 2100/450/450.


In [ ]:
!python -m prahari.ml.training.prepare_data --out models/dataset


## 4 · Train the LoRA adapters

r=16, α=32 on the attention projections, 4 epochs, seed fixed so the run is
reproducible. **This is the long step.**

The best checkpoint is selected on **recall, not F1** — deliberately. A false
positive costs one HSE review; a false negative costs a life. Expect
precision to look mediocre next to recall and do not "fix" it.

If Colab hands you a slower GPU, drop `--batch-size` to 8 rather than cutting
epochs.


In [ ]:
!python -m prahari.ml.training.train_lora \
    --dataset models/dataset \
    --out models/lora-adapters \
    --epochs 4 --batch-size 16 --seed 42


## 5 · Evaluate — including end to end, through the rule engine

Two things get measured here, and the second is the one that matters:

1. **Per-entity** precision / recall / F1 for the six span types.
2. **End-to-end SIF classification** after the rule engine runs on those
   spans — the number that is actually comparable to the keyword path's
   **83.3%** on this same test split.

It also prints a **false-negative dossier**: every missed SIF precursor with
its text. Read it. Those are the reports the system would have let through,
and showing them to judges deliberately is a stronger position than hoping
nobody asks.

`tests/ml/test_recall_floor.py` fails the build below **0.90 recall** — if
this run lands under that, the model does not ship.


In [ ]:
!python -m prahari.ml.training.evaluate \
    --dataset models/dataset --split test \
    --json-out models/eval_test.json


## 6 · Merge, quantise, export to ONNX

Merges the adapters into the base weights, quantises to INT8 and writes
`models/prahari.onnx` plus the tokenizer. The result runs on CPU, offline,
in well under the 200 ms/report budget.

Add `--no-quantise` only if the INT8 model loses meaningful recall — check
step 5's numbers against a re-run before deciding.


In [ ]:
!python -m prahari.ml.training.export_onnx \
    --adapters models/lora-adapters --out-dir models

!ls -la models/


## 7 · Download

Zips everything the runtime needs. Unzip into the repo's `models/` folder on
your laptop, then run `make verify` — the checksum section will finally have
something to check, and the startup banner should switch from
`keyword fallback` to `ONNX model`.

Keep `eval_test.json`. It is the evidence for every number you quote.


In [ ]:
import shutil, os
shutil.make_archive('/content/prahari-models', 'zip', '/content/prahari/models')
print('%.1f MB' % (os.path.getsize('/content/prahari-models.zip') / 1e6))

from google.colab import files
files.download('/content/prahari-models.zip')


---

### If something goes wrong

| Symptom | Cause |
|---|---|
| `CUDA out of memory` in step 4 | Drop `--batch-size` to 8. Keep the epochs. |
| Recall below 0.90 in step 5 | Try `--epochs 6`. If it is still short, the corpus vocabulary is the limit, not the model. |
| Session died mid-training | Colab idle-disconnect. Re-run from step 2; nothing is cached. |
| `prahari` not importable | Step 2's `%cd` did not take. Re-run that cell. |

**The demo works without any of this.** If training fails on the night, the
keyword extractor is still there, still scores 83.3%, and the system logs a
warning and carries on. That fallback is a design decision, not a
consolation prize.
